# LIME internals — verifying the claims behind the lecture

Technical companion to `lime_walkthrough.ipynb`. No teaching figures: only
commented code, prints, and small tables. It answers five questions the
lecture asserts but does not prove.

1. Does our description match the installed source of the `lime` package —
   perturbation, kernel, feature selection? (§5–§8)
2. How was the lecture's patient chosen, and how special is she? (§3)
3. Is LIME's synthetic neighborhood made of possible patients? (§4)
4. Does a high $R^2$ mean a better explanation? (§9)
5. How stable is an explanation across runs? (§10)

Several answers here contradict what an earlier draft of this material
claimed. Where that happens it is stated explicitly, because the wrong
version was the intuitive one and is worth inoculating against.

Runtime: §3 fits one explanation per test patient (143 × 5,000 samples).
Expect a few minutes.

## 1 · Setup

In [1]:
%pip install -q lime scikit-learn numpy scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from scipy.stats import spearmanr

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, roc_auc_score

from lime.lime_tabular import LimeTabularExplainer

RANDOM_STATE = 42

data = load_breast_cancer()
X_all, y_all = data.data, data.target
feature_names = list(data.feature_names)
class_names = list(data.target_names)
feat_idx = {f: i for i, f in enumerate(feature_names)}

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=RANDOM_STATE, stratify=y_all
)
model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE)
model.fit(X_train, y_train)
proba_test = model.predict_proba(X_test)[:, 1]
train_mean, train_std = X_train.mean(axis=0), X_train.std(axis=0)
kernel_width = 0.75 * np.sqrt(len(feature_names))  # lime_tabular.py:243 — the default

print(f"test accuracy = {accuracy_score(y_test, model.predict(X_test)):.1%}   "
      f"ROC-AUC = {roc_auc_score(y_test, proba_test):.3f}")


def fresh_explainer(seed=RANDOM_STATE):
    """A new explainer per call. This matters: reusing one explainer across
    many patients advances its internal RNG, so results drift between calls
    — a trap that produced a wrong claim in an earlier draft (see §3)."""
    return LimeTabularExplainer(
        X_train, feature_names=feature_names, class_names=class_names,
        discretize_continuous=False, random_state=seed,
    )


def explain(idx_or_row, seed=RANDOM_STATE):
    r = X_test[idx_or_row] if np.isscalar(idx_or_row) else idx_or_row
    return fresh_explainer(seed).explain_instance(
        r, model.predict_proba, num_features=8, num_samples=5000, labels=(1,))

test accuracy = 95.1%   ROC-AUC = 0.993


## 2 · Geometry helpers

One formula is worth stating before anything else, because it governs what
can and cannot appear in the lecture's figures. In standardized space the
fit's $P=0.5$ contour is $w_x z_x + w_y z_y + c = 0.5$, so its distance
from the patient is

$$\frac{\lvert g(x) - 0.5 \rvert}{\lVert (w_x, w_y) \rVert}$$

The numerator is how far the surrogate's own prediction sits from 0.5. For
a confidently classified patient it is large, and no choice of axes brings
that contour near her. Note the consequence, which cuts against us: any
patient *selected* for being near the boundary will automatically show a
nearby contour. That closeness is therefore a property of the filter, not
evidence that LIME recovered anything.

Implementation detail that matters throughout: the package fits on
STANDARDIZED data — `lime_tabular.py:452-454` passes `scaled_data`
(line 348) into `explain_instance_with_data`. So `local_exp`/`intercept`
live in standardized space.

In [3]:
def boundary_slice(base_row, ix, iy, grid_x, grid_y):
    GX, GY = np.meshgrid(grid_x, grid_y)
    pts = np.tile(base_row, (GX.size, 1))
    pts[:, ix] = GX.ravel()
    pts[:, iy] = GY.ravel()
    return GX, GY, model.predict_proba(pts)[:, 1].reshape(GX.shape)


def dist_to_boundary(base_row, ix, iy, span=3.0, n=120):
    gx = np.linspace(base_row[ix] - span * train_std[ix], base_row[ix] + span * train_std[ix], n)
    gy = np.linspace(base_row[iy] - span * train_std[iy], base_row[iy] + span * train_std[iy], n)
    GX, GY, P = boundary_slice(base_row, ix, iy, gx, gy)
    side = P >= 0.5
    if side.all() or (~side).all():
        return np.nan
    edge = np.zeros_like(side)
    edge[:, :-1] |= side[:, :-1] != side[:, 1:]
    edge[:-1, :] |= side[:-1, :] != side[1:, :]
    return float(np.min(np.hypot((GX[edge] - base_row[ix]) / train_std[ix],
                                 (GY[edge] - base_row[iy]) / train_std[iy])))


def pick_axes(exp, max_corr=0.6):
    """The rule the lecture actually uses: the highest-weight feature, plus
    the highest-ranked feature among the remaining selected ones that is not
    strongly correlated with it. Returns None if no such pair exists."""
    ranked = [f for f, _ in sorted(exp.local_exp[1], key=lambda t: abs(t[1]), reverse=True)]
    first = ranked[0]
    for cand in ranked[1:]:
        if abs(np.corrcoef(X_train[:, first], X_train[:, cand])[0, 1]) < max_corr:
            return first, cand
    return None

## 3 · How the lecture's patient was chosen — and a correction

An earlier draft claimed the patient was picked by requiring that her **two
highest-weight features** be decorrelated, and that "exactly one of 143
patients" satisfied the filter. Both halves were wrong, and the sweep below
shows why.

The rank-2 feature is almost always another size measurement, correlated
above 0.9 with rank 1, so a strict top-2 rule eliminates essentially
everybody. The lecture's axes are in fact ranks **1 and 3**. And the "one
survivor" figure came from a sweep that reused a single explainer, whose
RNG advances from patient to patient; with a fresh explainer per patient
the count changes. Sampling noise, presented as a property of the data.

The honest rule is `pick_axes` above, applied to a fresh explanation.

In [4]:
rows = []
for idx in range(len(X_test)):
    cand, p = X_test[idx], proba_test[idx]
    e = explain(idx)
    ranked = sorted(e.local_exp[1], key=lambda t: abs(t[1]), reverse=True)
    top2_corr = abs(np.corrcoef(X_train[:, ranked[0][0]], X_train[:, ranked[1][0]])[0, 1])
    pair = pick_axes(e)
    rows.append({
        "idx": idx, "P": p, "correct": bool((p >= 0.5) == y_test[idx]),
        "r2": e.score, "gx": e.local_pred[0],
        "top2_corr": top2_corr, "pair": pair,
        "rank_of_pair2": ([f for f, _ in ranked].index(pair[1]) + 1) if pair else None,
    })

strict = [r for r in rows if r["correct"] and abs(r["P"] - 0.5) < 0.15 and r["top2_corr"] < 0.6]
honest = [r for r in rows if r["correct"] and abs(r["P"] - 0.5) < 0.15 and r["pair"] is not None]

print(f"correctly classified and near the boundary: "
      f"{sum(r['correct'] and abs(r['P']-0.5) < 0.15 for r in rows)} of {len(rows)}")
print(f"  ... whose TOP-2 are decorrelated (the wrong, discarded rule): {len(strict)}")
print(f"  ... for which SOME decorrelated pair exists (the rule used):  {len(honest)}")
print(f"\nmedian correlation between rank-1 and rank-2 features, all patients: "
      f"{np.median([r['top2_corr'] for r in rows]):.2f}")
print("→ rank 2 is nearly always a near-duplicate of rank 1. The plotting pair has to")
print("  come from further down the list; that is a visualization decision, not a finding.")
print("\ncandidates the honest rule admits:")
for r in honest:
    a, b = r["pair"]
    print(f"  #{r['idx']:3d}  P={r['P']:.3f}  R²={r['r2']:.2f}  "
          f"axes = {feature_names[a]} (rank 1) + {feature_names[b]} (rank {r['rank_of_pair2']})")
print("\nThe lecture uses #67. Note it does NOT have the best R² of the four — that")
print("would be the wrong tie-breaker anyway (§9). It is chosen because its plottable")
print("second axis sits highest in the ranking (rank 3, against ranks 4-5), so the")
print("figure shows the two most important features that can honestly share a plot.")
print("Any of the four would serve; there is nothing unique about her.")

correctly classified and near the boundary: 4 of 143
  ... whose TOP-2 are decorrelated (the wrong, discarded rule): 0
  ... for which SOME decorrelated pair exists (the rule used):  4

median correlation between rank-1 and rank-2 features, all patients: 0.98
→ rank 2 is nearly always a near-duplicate of rank 1. The plotting pair has to
  come from further down the list; that is a visualization decision, not a finding.

candidates the honest rule admits:
  #  2  P=0.629  R²=0.56  axes = worst area (rank 1) + worst concavity (rank 5)
  # 10  P=0.642  R²=0.44  axes = worst perimeter (rank 1) + worst texture (rank 5)
  # 67  P=0.581  R²=0.36  axes = worst perimeter (rank 1) + worst texture (rank 3)
  #111  P=0.564  R²=0.52  axes = worst area (rank 1) + worst texture (rank 4)

The lecture uses #67. Note it does NOT have the best R² of the four — that
would be the wrong tie-breaker anyway (§9). It is chosen because its plottable
second axis sits highest in the ranking (rank 3, against ranks

In [5]:
instance_idx = 67
row = X_test[instance_idx]
row_proba = proba_test[instance_idx]
row_scaled = (row - train_mean) / train_std
ix, iy = feat_idx["worst perimeter"], feat_idx["worst texture"]
print(f"lecture patient #{instance_idx}: P(benign)={row_proba:.3f}, "
      f"true={class_names[y_test[instance_idx]]}")
print(f"boundary distance on the plotted axes: {dist_to_boundary(row, ix, iy):.2f}σ")
print("(reminder from §2: small by construction — she was selected for it)")

lecture patient #67: P(benign)=0.581, true=benign
boundary distance on the plotted axes: 0.08σ
(reminder from §2: small by construction — she was selected for it)


## 4 · Is the synthetic neighborhood made of possible patients?

LIME draws each feature independently from its marginal. Real tumour
measurements are strongly dependent — perimeter, radius and area are three
views of one geometry. Two checks: how many synthetic patients are
physically impossible outright, and whether a basic geometric identity
survives.

In [6]:
rng = np.random.RandomState(RANDOM_STATE + 1)
Z_probe = train_mean + train_std * rng.normal(0, 1, size=(5000, len(feature_names)))

neg_any = (Z_probe < 0).any(axis=1).mean()
neg_feats = sorted(((Z_probe[:, j] < 0).mean(), feature_names[j]) for j in range(len(feature_names)))[-4:]
ratio_syn = Z_probe[:, feat_idx["worst perimeter"]] / Z_probe[:, feat_idx["worst radius"]]
ratio_real = X_all[:, feat_idx["worst perimeter"]] / X_all[:, feat_idx["worst radius"]]

print(f"synthetic patients with at least one negative measurement: {neg_any:.1%}")
print("most frequently negative features:")
for frac, name in reversed(neg_feats):
    print(f"    {name:<26} negative in {frac:.0%} of draws")
print(f"\nperimeter / radius ratio (≈2π = {2*np.pi:.2f} for any closed shape)")
print(f"  real patients:      [{ratio_real.min():.2f}, {ratio_real.max():.2f}]")
print(f"  synthetic (1–99%):  [{np.percentile(ratio_syn,1):.2f}, {np.percentile(ratio_syn,99):.2f}]")
print("\n→ the neighborhood LIME reasons over is not a set of plausible patients.")
print("  Every explanation in this material is an explanation of how the model")
print("  behaves on impossible data near the patient. That is the method, not a bug")
print("  in our setup — but it is rarely stated this plainly.")

synthetic patients with at least one negative measurement: 75.6%
most frequently negative features:
    area error                 negative in 18% of draws
    concavity error            negative in 17% of draws
    mean concavity             negative in 14% of draws
    worst concavity            negative in 11% of draws

perimeter / radius ratio (≈2π = 6.28 for any closed shape)
  real patients:      [6.22, 7.67]
  synthetic (1–99%):  [1.82, 21.50]

→ the neighborhood LIME reasons over is not a set of plausible patients.
  Every explanation in this material is an explanation of how the model
  behaves on impossible data near the patient. That is the method, not a bug
  in our setup — but it is rarely stated this plainly.


## 5 · The official explanation, spied on

`predict_proba` is swapped for a spy that records the neighborhood the
package generates internally. §6–§8 reuse exactly that sample.

In [7]:
captured = {}


def spy(X_in):
    captured["X"] = np.array(X_in, dtype=float).copy()
    return model.predict_proba(X_in)


exp_official = fresh_explainer().explain_instance(
    row, spy, num_features=8, num_samples=5000, labels=(1,))
Z_cap = captured["X"]
proba_cap = model.predict_proba(Z_cap)[:, 1]
print(f"captured: {Z_cap.shape[0]} samples × {Z_cap.shape[1]} features   "
      f"official R² = {exp_official.score:.3f}")

captured: 5000 samples × 30 features   official R² = 0.357


## 6 · Perturbation vs `__data_inverse`

`lime_tabular.py:511-517`: draw $\mathcal{N}(0,1)$ per feature and undo the
standardization — `data = data*scale + mean`, with the TRAINING mean/std
(`StandardScaler(with_mean=False)`, lines 257-258). This holds because
`sample_around_instance=False` is the default (line 138); otherwise it
would be `+ instance_sample` (line 515).

The consequence deserves emphasis: **the cloud is centred on the dataset,
not on the patient.** Every patient gets the same cloud; only the weights
differ. §9 turns on this fact.

In [8]:
print(f"largest deviation of captured mean from train_mean: "
      f"{np.abs((Z_cap.mean(0) - train_mean) / train_std).max():.1%} of σ")
print(f"largest deviation of captured std  from train_std:  "
      f"{np.abs((Z_cap.std(0) - train_std) / train_std).max():.1%} of σ")
print(f"distance from the cloud's centre to the patient: "
      f"{np.linalg.norm(row_scaled):.2f}σ  (the cloud is centred at 0 by construction)")
print("→ deviations are sampling noise at n=5,000 in 30-D. The formula matches.")

largest deviation of captured mean from train_mean: 3.0% of σ
largest deviation of captured std  from train_std:  3.6% of σ
distance from the cloud's centre to the patient: 2.95σ  (the cloud is centred at 0 by construction)
→ deviations are sampling noise at n=5,000 in 30-D. The formula matches.


## 7 · The proximity kernel

`lime_tabular.py:243`: $\nu = 0.75\sqrt{p}$. `lime_tabular.py:248`:
$\pi = \sqrt{\exp(-d^2/\nu^2)} = \exp(-d^2/2\nu^2)$ — a Gaussian with
standard deviation exactly $\nu$. (An earlier draft said $\nu\sqrt2$; that
was wrong.) Distances use the standardized features.

In [9]:
Z_scaled = (Z_cap - train_mean) / train_std
dist = np.linalg.norm(Z_scaled - row_scaled, axis=1)
weight = np.sqrt(np.exp(-(dist ** 2) / kernel_width ** 2))
print(f"ν = {kernel_width:.2f} (30-D standardized units)")
print(f"nearest 'neighbor': {dist.min():.2f}σ → weight {weight.max():.3f}")
print("  (that is the patient herself: lime_tabular.py:518 sets data[0] = the instance,")
print("   so the sample always contains one point at distance 0 with weight exactly 1)")
print(f"median neighbor:  {np.median(dist):.2f}σ → weight {np.median(weight):.3f}, "
      f"i.e. {np.median(weight)/weight.max():.0%} of the maximum")
print(f"effective sample size (Σw)²/Σw² = {weight.sum()**2/(weight**2).sum():.0f} "
      f"of {len(weight)} draws")
print("→ 'local' is very wide. The typical contributing point is several σ away, and")
print("  the weights are close to uniform: the effective sample size is barely below")
print("  the nominal one. The kernel is doing much less localizing than its name suggests.")

ν = 4.11 (30-D standardized units)
nearest 'neighbor': 0.00σ → weight 1.000
  (that is the patient herself: lime_tabular.py:518 sets data[0] = the instance,
   so the sample always contains one point at distance 0 with weight exactly 1)
median neighbor:  6.18σ → weight 0.323, i.e. 32% of the maximum
effective sample size (Σw)²/Σw² = 4645 of 5000 draws
→ 'local' is very wide. The typical contributing point is several σ away, and
  the weights are close to uniform: the effective sample size is barely below
  the nominal one. The kernel is doing much less localizing than its name suggests.


## 8 · The `highest_weights` selection, and a trap

`lime_base.py:109`: `weighted_data = coef * data[0]` — an auxiliary Ridge
($\alpha=0.01$) over all features, weighted by the kernel; score each
feature by $\lvert\text{coef}\times\text{value}\rvert$; keep the top
`num_features`.

The trap: `data[0]` is the **standardized** patient, not her raw values.
Reproducing this in raw units silently yields a different feature set. We
made exactly that mistake while building this material, and it corrupted
the plotted geometry before it was caught — so both versions are run here.

In [10]:
aux = Ridge(alpha=0.01).fit(Z_scaled, proba_cap, sample_weight=weight)
top8_std = set(np.argsort(-np.abs(aux.coef_ * row_scaled))[:8])
top8_raw = set(np.argsort(-np.abs(aux.coef_ * row))[:8])
top8_off = set(f for f, _ in exp_official.local_exp[1])
print(f"standardized (correct) matches the package: {top8_std == top8_off}  "
      f"({len(top8_std & top8_off)}/8)")
print(f"raw units (the bug)     matches the package: {top8_raw == top8_off}  "
      f"({len(top8_raw & top8_off)}/8)")
print("\nofficial coefficients (negative = pushes toward malignant):")
for f, w in sorted(exp_official.local_exp[1], key=lambda t: abs(t[1]), reverse=True):
    print(f"  {feature_names[f]:<26} {w:+.4f}")
print("\n`lime_base.py:194-196` confirms the reported score is weighted; "
      "`:204-206` that local_exp is sorted by |w|.")

standardized (correct) matches the package: True  (8/8)
raw units (the bug)     matches the package: False  (5/8)

official coefficients (negative = pushes toward malignant):
  worst perimeter            -0.0709
  worst concave points       -0.0457
  worst texture              -0.0213
  area error                 -0.0184
  mean texture               -0.0153
  mean perimeter             -0.0140
  mean radius                -0.0134
  radius error               -0.0089

`lime_base.py:194-196` confirms the reported score is weighted; `:204-206` that local_exp is sorted by |w|.


## 9 · Does a high $R^2$ mean a better explanation?

The observed regularity first, then the explanation that fails, then what
is left standing.

In [11]:
conf = np.array([abs(r["P"] - 0.5) for r in rows])
fid = np.array([r["r2"] for r in rows])
rho, pv = spearmanr(conf, fid)
print(f"Spearman(|P-0.5|, R²) = {rho:+.3f}   p = {pv:.1e}   n = {len(rows)}")
for lo, hi, lab in [(0.0, 0.15, "borderline"), (0.15, 0.30, "intermediate"), (0.30, 0.51, "confident")]:
    s = (conf >= lo) & (conf < hi)
    if s.sum():
        print(f"  {lab:14s} n={s.sum():3d}   mean R² = {fid[s].mean():.3f}")

Spearman(|P-0.5|, R²) = +0.612   p = 4.4e-16   n = 143
  borderline     n=  6   mean R² = 0.472
  intermediate   n= 10   mean R² = 0.565
  confident      n=127   mean R² = 0.614


**The intuitive explanation, and why it is false.** One would say: far from
the boundary the model saturates, its output barely varies, and a line
reproduces "almost constant" easily. Three problems.

In [12]:
f_probe = model.predict_proba(Z_probe)[:, 1]
f_real = model.predict_proba(X_all)[:, 1]
print("(a) does the model saturate where LIME looks?")
print(f"    f on the synthetic cloud: [{f_probe.min():.2f}, {f_probe.max():.2f}], "
      f"fraction <0.05 or >0.95 = {((f_probe<.05)|(f_probe>.95)).mean():.3f}")
print(f"    f on real patients:       fraction <0.05 or >0.95 = "
      f"{((f_real<.05)|(f_real>.95)).mean():.1%}")
print("    → never. The saturated region is where the real patients are, and §4")
print("      showed the cloud is nowhere near them.")

wvar, cent, skew = [], [], []
for i in range(len(X_test)):
    dd = np.linalg.norm((Z_probe - train_mean) / train_std - (X_test[i] - train_mean) / train_std, axis=1)
    ww = np.sqrt(np.exp(-(dd ** 2) / kernel_width ** 2))
    wvar.append(np.average((f_probe - np.average(f_probe, weights=ww)) ** 2, weights=ww))
    cent.append(np.linalg.norm((X_test[i] - train_mean) / train_std))
    skew.append(ww.max() / ww.min())
wvar, cent, skew = np.array(wvar), np.array(cent), np.array(skew)
b, c = conf < 0.15, conf > 0.30
print("\n(b) is the target flatter for confident patients?")
print(f"    weighted Var(f):  borderline {wvar[b].mean():.4f}   confident {wvar[c].mean():.4f}   "
      f"ρ(conf, var) = {spearmanr(conf, wvar)[0]:+.2f}")
print("    → no: identical. They share one cloud; only the weights differ.")
print("\n(c) the algebra points the other way:")
print("    R² = 1 − SSE/SST. A flatter target shrinks SST, making high R² HARDER.")
print("    The intuitive story predicts the opposite of the observed correlation.")

print("\n(d) the candidate replacement explanation — also not supported:")
print(f"    distance from the training centroid: borderline {cent[b].mean():.2f}σ   "
      f"confident {cent[c].mean():.2f}σ   ρ = {spearmanr(conf, cent)[0]:+.2f}")
print(f"    kernel weight skew (max/min):        borderline {np.median(skew[b]):.0f}   "
      f"confident {np.median(skew[c]):.0f}   ρ = {spearmanr(conf, skew)[0]:+.2f}")
print("    Confident patients do sit further from the centroid. But the obvious next")
print("    step — 'so the kernel concentrates weight on fewer points, making the fit")
print("    easier' — does not hold either: the skew barely moves, and §7 showed the")
print("    effective sample size is ~93% of the nominal one. The weights are close to")
print("    uniform for everybody.")
print("\nSo: the correlation is robust and replicated, and we cannot say why.")
print("Both mechanisms one would reach for are falsified above. Stating that honestly")
print("is better teaching than a tidy story that does not survive its own measurement.")
print("\nBottom line for practice, unchanged by the open mechanism:")
print("  R² is not an explanation-quality score. Do not rank explanations by it.")

(a) does the model saturate where LIME looks?
    f on the synthetic cloud: [0.07, 0.97], fraction <0.05 or >0.95 = 0.000
    f on real patients:       fraction <0.05 or >0.95 = 73.8%
    → never. The saturated region is where the real patients are, and §4
      showed the cloud is nowhere near them.

(b) is the target flatter for confident patients?
    weighted Var(f):  borderline 0.0228   confident 0.0227   ρ(conf, var) = -0.05
    → no: identical. They share one cloud; only the weights differ.

(c) the algebra points the other way:
    R² = 1 − SSE/SST. A flatter target shrinks SST, making high R² HARDER.
    The intuitive story predicts the opposite of the observed correlation.

(d) the candidate replacement explanation — also not supported:
    distance from the training centroid: borderline 2.70σ   confident 4.84σ   ρ = +0.30
    kernel weight skew (max/min):        borderline 12   confident 13   ρ = +0.12
    Confident patients do sit further from the centroid. But the obvious 

## 10 · Direction vs level, and stability

A linear fit offers two things: a direction (coefficient ratios) and a
level (the intercept). They are not equally trustworthy, and the lecture's
figure depends on the difference.

In [13]:
def numeric_gradient(base_row, h=0.3):
    g = np.zeros(len(feature_names))
    for j in range(len(feature_names)):
        a, bb = base_row.copy(), base_row.copy()
        a[j] += h * train_std[j]
        bb[j] -= h * train_std[j]
        g[j] = (model.predict_proba(a.reshape(1, -1))[0, 1]
                - model.predict_proba(bb.reshape(1, -1))[0, 1]) / (2 * h)
    return g


gf = numeric_gradient(row)
cos = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
angles, gxs, cos2, cos30, sets = [], [], [], [], []
for s in range(8):
    e = explain(instance_idx, seed=RANDOM_STATE + s)
    cm = dict(e.local_exp[1])
    gg = np.zeros(len(feature_names))
    for j, w in cm.items():
        gg[j] = w
    angles.append(np.degrees(np.arctan2(gg[iy], gg[ix])))
    gxs.append(e.local_pred[0])
    cos2.append(cos(gg[[ix, iy]], gf[[ix, iy]]))
    cos30.append(cos(gg, gf))
    sets.append(set(cm))
angles, gxs = np.array(angles), np.array(gxs)

print(f"DIRECTION, in the plotted plane:  {np.mean(angles):+.1f}° ± {np.std(angles):.1f}°   "
      f"vs the model's own {np.degrees(np.arctan2(gf[iy], gf[ix])):+.1f}°")
print(f"  cosine with the model's local gradient: {np.mean(cos2):+.3f} (2-D), "
      f"{np.mean(cos30):+.3f} (all 30-D)")
print(f"LEVEL:  g(x) = {gxs.mean():.4f} ± {gxs.std():.4f}   vs f(x) = {row_proba:.4f}")
print(f"  correct side of 0.5 in {int((gxs>=0.5).sum())}/8 runs")
print(f"FEATURE SET: {np.mean([len(s & sets[0]) for s in sets[1:]]):.1f}/8 shared with run 1 on average")
print("\n→ the direction is precise in the plane we plot and only moderate across all 30")
print("  features; the level is stably WRONG (a bias, not noise: σ≈0.003 around a value")
print("  0.08 away from the truth). The intercept is dragged toward the mean prediction")
print("  of the off-manifold cloud. This is why the lecture draws the fit through the")
print("  patient instead of at P=0.5 — the P=0.5 contour would put her on the malignant")
print("  side of her own explanation.")

DIRECTION, in the plotted plane:  -162.7° ± 1.4°   vs the model's own -167.4°
  cosine with the model's local gradient: +0.996 (2-D), +0.597 (all 30-D)
LEVEL:  g(x) = 0.4988 ± 0.0025   vs f(x) = 0.5812
  correct side of 0.5 in 3/8 runs
FEATURE SET: 7.1/8 shared with run 1 on average

→ the direction is precise in the plane we plot and only moderate across all 30
  features; the level is stably WRONG (a bias, not noise: σ≈0.003 around a value
  0.08 away from the truth). The intercept is dragged toward the mean prediction
  of the off-manifold cloud. This is why the lecture draws the fit through the
  patient instead of at P=0.5 — the P=0.5 contour would put her on the malignant
  side of her own explanation.


## Summary

- Perturbation, kernel and feature selection reproduce the installed `lime`
  source exactly, provided the selection is done in standardized space (§8).
- The lecture's axes are ranks 1 and 3, chosen for legibility; rank 2 is
  almost always a near-duplicate of rank 1, and the earlier "one patient in
  143" claim was RNG noise (§3).
- LIME's neighborhood is not made of possible patients: most draws carry a
  negative measurement and basic geometry breaks (§4).
- $R^2$ tracks prediction confidence, the saturation explanation for it is
  false, and the mechanism remains only partly identified (§9).
- A local fit's direction is trustworthy; its level is not (§10).

Molnar lists several limitations of LIME; three appear quantitatively
above — neighborhood definition (§7), sampling that ignores feature
correlation (§4), and instability across runs (§10).

In [14]:
print("Lecture version, with the figures: lime_walkthrough.ipynb")

Lecture version, with the figures: lime_walkthrough.ipynb
